# 1. Voting Classifier
#### In this assignment, you are expected to build an ensemble of different models and train it on cover type dataset.

## 1.1. Load dataset
#### You will need to read the data from the file (cover.csv). It contains 581012 samples and 54 attributes for each sample. The target column is Cover_Type.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print("="*60)
print("PART 1: VOTING CLASSIFIER")
print("="*60)

## 1.1. Load dataset
print("\n1.1 Loading cover.csv dataset...")
# Load the cover type dataset
cover_data = pd.read_csv('cover.csv')
print(f"Dataset shape: {cover_data.shape}")
print(f"Target column unique values: {cover_data['Cover_Type'].unique()}")
print(f"Dataset info:")
print(cover_data.info())

# Separate features and target
X_cover = cover_data.drop('Cover_Type', axis=1)
y_cover = cover_data['Cover_Type']

## 1.2. Prepare dataset
#### Split the data into train, validation, and test sets using train_test_split twice with 0.2 test_size. Your final distribution will be 371847-92962-116203.

In [5]:
print("\n1.2 Splitting dataset...")
# First split: 80% train+val, 20% test
X_temp, X_test_cover, y_temp, y_test_cover = train_test_split(
    X_cover, y_cover, test_size=0.2, random_state=42, stratify=y_cover
)

# Second split: 80% train, 20% val (of the remaining 80%)
X_train_cover, X_val_cover, y_train_cover, y_val_cover = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Training set size: {X_train_cover.shape[0]}")
print(f"Validation set size: {X_val_cover.shape[0]}")
print(f"Test set size: {X_test_cover.shape[0]}")


1.2 Splitting dataset...
Training set size: 371847
Validation set size: 92962
Test set size: 116203


## 1.3. Modeling
#### Train 4-5 different classifiers on the data. You can train RandomForestClassifier, ExtraTreesClassifier, LinearSVC, SGDClassifier, MLPClassifier, etc. Evaluate their performances using validation set. Note that training may take quite a while (up to 30 minutes) depending on the hardware.

In [ ]:
print("\n1.3 Training individual classifiers...")

# Define classifiers
classifiers = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'LinearSVC': LinearSVC(random_state=42, max_iter=1000),
    'SGD': SGDClassifier(random_state=42, max_iter=1000),
    'MLP': MLPClassifier(random_state=42, max_iter=300, hidden_layer_sizes=(100,))
}

# Train and evaluate individual classifiers
individual_scores = {}
trained_classifiers = {}

for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train_cover, y_train_cover)
    
    # Predict on validation set
    val_pred = clf.predict(X_val_cover)
    val_score = accuracy_score(y_val_cover, val_pred)
    
    individual_scores[name] = val_score
    trained_classifiers[name] = clf
    
    print(f"{name} validation accuracy: {val_score:.4f}")

print("\nIndividual classifier performances:")
for name, score in individual_scores.items():
    print(f"{name}: {score:.4f}")


1.3 Training individual classifiers...

Training RandomForest...
RandomForest validation accuracy: 0.9487

Training ExtraTrees...
ExtraTrees validation accuracy: 0.9480

Training LinearSVC...
LinearSVC validation accuracy: 0.7022

Training SGD...
SGD validation accuracy: 0.6694

Training MLP...


## 1.4. Ensembling
#### Create a hard and soft voting classifier using the models you have trained. You can use VotingClassifier. Check its performance on the validation set. Do you get better or worse performance than any of the individual classifiers?

In [ ]:
print("\n1.4 Creating voting classifiers...")

# Prepare estimators for voting classifier
estimators = [(name, clf) for name, clf in trained_classifiers.items()]

# Hard voting classifier
print("\nTraining Hard Voting Classifier...")
hard_voting_clf = VotingClassifier(estimators=estimators, voting='hard')
hard_voting_clf.fit(X_train_cover, y_train_cover)
hard_val_pred = hard_voting_clf.predict(X_val_cover)
hard_val_score = accuracy_score(y_val_cover, hard_val_pred)
print(f"Hard Voting validation accuracy: {hard_val_score:.4f}")

# Soft voting classifier (only for classifiers that support predict_proba)
soft_estimators = []
for name, clf in trained_classifiers.items():
    if hasattr(clf, 'predict_proba'):
        soft_estimators.append((name, clf))
    else:
        print(f"Note: {name} doesn't support predict_proba, excluding from soft voting")

if len(soft_estimators) > 1:
    print("\nTraining Soft Voting Classifier...")
    soft_voting_clf = VotingClassifier(estimators=soft_estimators, voting='soft')
    soft_voting_clf.fit(X_train_cover, y_train_cover)
    soft_val_pred = soft_voting_clf.predict(X_val_cover)
    soft_val_score = accuracy_score(y_val_cover, soft_val_pred)
    print(f"Soft Voting validation accuracy: {soft_val_score:.4f}")
else:
    print("Not enough classifiers support predict_proba for soft voting")
    soft_val_score = None

# Compare ensemble vs individual performance
print("\nPerformance Comparison:")
print(f"Best individual classifier: {max(individual_scores, key=individual_scores.get)} - {max(individual_scores.values()):.4f}")
print(f"Hard Voting Ensemble: {hard_val_score:.4f}")
if soft_val_score:
    print(f"Soft Voting Ensemble: {soft_val_score:.4f}")

# Check if any model hurts ensemble performance
print("\nAnalyzing individual estimator contributions...")
best_ensemble = hard_voting_clf if not soft_val_score or hard_val_score > soft_val_score else soft_voting_clf
best_ensemble_score = hard_val_score if not soft_val_score or hard_val_score > soft_val_score else soft_val_score

# Try removing each estimator and see if performance improves
print("Testing ensemble without each estimator:")
for i, (name, _) in enumerate(best_ensemble.estimators):
    temp_estimators = [est for j, est in enumerate(best_ensemble.estimators) if j != i]
    temp_voting_clf = VotingClassifier(estimators=temp_estimators, voting=best_ensemble.voting)
    temp_voting_clf.fit(X_train_cover, y_train_cover)
    temp_pred = temp_voting_clf.predict(X_val_cover)
    temp_score = accuracy_score(y_val_cover, temp_pred)
    
    improvement = temp_score - best_ensemble_score
    print(f"Without {name}: {temp_score:.4f} (change: {improvement:+.4f})")

#### Check if any of the models hurts the performance of the ensemble. You can access the estimators of the ensemble using estimators_ attribute. If so, drop those using set_params and reevaluate.

# 2. Random Forest
#### In this assignment, you are expected to build a random forest that classifies a toy dataset.

## 2.1. Load dataset
#### You will need to read the data from the file (data.csv). It contains 15000 samples and two features for each sample.

In [ ]:
print("\n" + "="*60)
print("PART 2: RANDOM FOREST")
print("="*60)

## 2.1. Load dataset
print("\n2.1 Loading data.csv dataset...")
toy_data = pd.read_csv('data.csv')
print(f"Dataset shape: {toy_data.shape}")
print("First few rows:")
print(toy_data.head())

# The CSV seems to have numerical column names, let's rename them
toy_data.columns = ['feature1', 'feature2', 'target']
X_toy = toy_data[['feature1', 'feature2']]
y_toy = toy_data['target']

print(f"Features shape: {X_toy.shape}")
print(f"Target unique values: {y_toy.unique()}")

## 2.2. Prepare dataset
#### Split the data into train and test sets with 0.2 test size.

In [ ]:
print("\n2.2 Splitting toy dataset...")
X_train_toy, X_test_toy, y_train_toy, y_test_toy = train_test_split(
    X_toy, y_toy, test_size=0.2, random_state=42, stratify=y_toy
)

print(f"Training set size: {X_train_toy.shape[0]}")
print(f"Test set size: {X_test_toy.shape[0]}")


## 2.3. Modeling
#### Train a DecisionTreeClassifier on the data. Use GridSearchCV to tune the hyperparameters.

In [ ]:
print("\n2.3 Training Decision Tree with GridSearchCV...")

# Define parameter grid for DecisionTreeClassifier
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# GridSearchCV
dt_clf = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(
    dt_clf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)

print("Performing grid search...")
grid_search.fit(X_train_toy, y_train_toy)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")


#### Train the best model on the whole train set (do you need to?) and evaluate the model on the test set.

In [ ]:
print("\nTraining best model on full training set...")
best_dt = grid_search.best_estimator_
# Note: GridSearchCV already trains on the full training set, so we don't need to refit
test_pred = best_dt.predict(X_test_toy)
test_score = accuracy_score(y_test_toy, test_pred)
print(f"Single Decision Tree test accuracy: {test_score:.4f}")

#### Generate 1,200 subsets of the training set, each containing 100 randomly chosen instances. You can use ShuffleSplit.

In [ ]:
print("\n2.4 Generating 1,200 subsets and training trees...")
n_subsets = 100
subset_size = 100
shuffle_split = ShuffleSplit(n_splits=n_subsets, train_size=subset_size, random_state=42)


#### Train one tree on each subset, using the best model you previously found. Evaluate the performance of the trees using the test set. Did you get lower or higher accuracy? Why?

In [ ]:
print("Training trees on subsets...")
subset_trees = []
subset_accuracies = []

for i, (train_idx, _) in enumerate(shuffle_split.split(X_train_toy)):
    if i % 50 == 0:
        print(f"Training tree {i+1}/{n_subsets}")
    
    # Get subset
    X_subset = X_train_toy.iloc[train_idx]
    y_subset = y_train_toy.iloc[train_idx]
    
    # Train tree with best parameters found by GridSearchCV
    tree = DecisionTreeClassifier(**grid_search.best_params_)
    tree.fit(X_subset, y_subset)
    
    # Evaluate on test set
    subset_pred = tree.predict(X_test_toy)
    subset_acc = accuracy_score(y_test_toy, subset_pred)
    
    subset_trees.append(tree)
    subset_accuracies.append(subset_acc)

print(f"\nResults:")
print(f"Average accuracy of individual trees: {np.mean(subset_accuracies):.4f}")
print(f"Standard deviation: {np.std(subset_accuracies):.4f}")
print(f"Best individual tree accuracy: {np.max(subset_accuracies):.4f}")
print(f"Worst individual tree accuracy: {np.min(subset_accuracies):.4f}")

# Answer the question: Did you get lower or higher accuracy? Why?
print(f"\nComparison:")
print(f"Single Decision Tree (full training set): {test_score:.4f}")
print(f"Average of individual trees (subsets): {np.mean(subset_accuracies):.4f}")

if np.mean(subset_accuracies) < test_score:
    print("✓ Individual trees have LOWER accuracy than the single tree")
    print("  This is expected because:")
    print("  - Each subset has only 100 samples vs ~12,000 in full training set")
    print("  - Less data = higher variance and potential underfitting")
    print("  - Individual trees see less diverse examples")
else:
    print("✗ Individual trees have HIGHER accuracy than the single tree")
    print("  This suggests the single tree might be overfitting")

#### For each instance in the test set, predict its class using 1200 trees, and keep only the most frequent prediction. You can use mode from scipy.stats. Evaluate these predictions. Did you get lower or higher accuracy?

In [ ]:
print("Creating ensemble predictions using majority voting...")

# Collect predictions from all trees
all_predictions = np.array([tree.predict(X_test_toy) for tree in subset_trees])
print(f"Prediction matrix shape: {all_predictions.shape}")  # Should be (1200, test_size)

# Use majority voting (mode) for final predictions
ensemble_predictions = []
for i in range(len(X_test_toy)):
    # Get predictions for this instance from all trees
    instance_predictions = all_predictions[:, i]
    # Find the mode (most frequent prediction)
    ensemble_pred = mode(instance_predictions, keepdims=False)[0]
    ensemble_predictions.append(ensemble_pred)

ensemble_predictions = np.array(ensemble_predictions)
ensemble_accuracy = accuracy_score(y_test_toy, ensemble_predictions)

print(f"\nFinal Results:")
print(f"Single Decision Tree: {test_score:.4f}")
print(f"Average of 1200 individual trees: {np.mean(subset_accuracies):.4f}")
print(f"Random Forest (ensemble of 1200 trees): {ensemble_accuracy:.4f}")

# Answer the question: Did you get lower or higher accuracy?
print(f"\nAnalysis:")
if ensemble_accuracy > test_score:
    print("✓ Random Forest achieved HIGHER accuracy than single tree")
    print("  Ensemble methods reduce variance through averaging")
    print("  Multiple weak learners combine to form a strong learner")
elif ensemble_accuracy > np.mean(subset_accuracies):
    print("✓ Random Forest achieved HIGHER accuracy than individual trees")
    print("  Even if not better than single tree, ensemble reduces overfitting")
else:
    print("✗ Random Forest achieved LOWER accuracy")
    print("  This might happen if trees are too weak or correlated")

print(f"\nVariance reduction effect:")
print(f"Individual tree variance: {np.std(subset_accuracies):.4f}")
print(f"Improvement from ensemble: {ensemble_accuracy - np.mean(subset_accuracies):+.4f}")